# LC 42 — Trapping Rain Water
**Difficulty:** Hard &nbsp;|&nbsp; **Category:** Two Pointers
**Pattern:** Two Pointers with Running Max — Left Meets Right

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Water above a bar equals
the minimum of the tallest bar to its left and the
tallest bar to its right, minus its own height.
Two pointers let you compute that minimum on the fly:
whichever side is shorter already fixes the water
ceiling — no need to look across the array.
</div>

## Official Problem Statement

Given `n` non-negative integers representing an
elevation map where the width of each bar is `1`,
compute how much water it can trap after raining.

**Example 1:**
```
Input:  height = [0,1,0,2,1,0,1,3,1,0,1,2]
Output: 6
```
**Example 2:**
```
Input:  height = [4,2,0,3,1,2,5]
Output: 9
```

**Constraints:**
- `n == height.length`
- `1 <= n <= 2 * 10^4`
- `0 <= height[i] <= 10^5`

## What This Is Actually Asking

Think of the height array as a cross-section of a
mountain range after rainfall.
Water pools wherever a bar is shorter than the tallest
bars on both sides of it.
For each bar, how deep does the pool get? It fills up
to the shorter of the two surrounding peaks, then
drains to the bar's own height.
Sum all those trapped amounts.

## Walk Through an Example by Hand

```
height = [0, 1, 0, 2, 1, 0, 1, 3, 1, 0, 1, 2]
index:    0  1  2  3  4  5  6  7  8  9 10 11

L=0  R=11   left_max=0  right_max=2
  h[L]=0 <= h[R]=2  ->  process LEFT
  h[L]=0 <= left_max=0  ->  water += 0-0 = 0   total=0   L++

L=1  R=11   left_max=0  right_max=2
  h[L]=1 <= h[R]=2  ->  process LEFT
  h[L]=1 > left_max=0  ->  update left_max=1, water += 0  L++

L=2  R=11   left_max=1  right_max=2
  h[L]=0 <= h[R]=2  ->  process LEFT
  h[L]=0 <= left_max=1  ->  water += 1-0 = 1   total=1   L++

L=3  R=11   left_max=1  right_max=2
  h[L]=2 <= h[R]=2  ->  process LEFT
  h[L]=2 > left_max=1  ->  update left_max=2, water += 0  L++

L=4  R=11   left_max=2  right_max=2
  h[L]=1 <= h[R]=2  ->  process LEFT
  h[L]=1 <= left_max=2  ->  water += 2-1 = 1   total=2   L++

L=5  R=11   left_max=2  right_max=2
  h[L]=0 <= h[R]=2  ->  process LEFT
  h[L]=0 <= left_max=2  ->  water += 2-0 = 2   total=4   L++

L=6  R=11   left_max=2  right_max=2
  h[L]=1 <= h[R]=2  ->  process LEFT
  h[L]=1 <= left_max=2  ->  water += 2-1 = 1   total=5   L++

L=7  R=11   left_max=2  right_max=2
  h[L]=3 > h[R]=2  ->  process RIGHT
  h[R]=2 >= right_max=2  ->  update right_max=2, water += 0  R--

L=7  R=10   left_max=2  right_max=2
  h[L]=3 > h[R]=1  ->  process RIGHT
  h[R]=1 <= right_max=2  ->  water += 2-1 = 1   total=6   R--

L=7  R=9    h[L]=3 > h[R]=0 -> process RIGHT
  water += 2-0 = 2  ... but L=7 >= R=9 eventually -> stop

Answer: 6
```

## The Picture

```
height = [0, 1, 0, 2, 1, 0, 1, 3, 1, 0, 1, 2]

                        |
         |              |           |
         |   W  |   W  W  W  |  W  |
   |  W  |   |  W  W  W  |  |  W  |
   0  1  0   2  1  0  1  3  1  0  1  2
   L                               R

W = trapped water cells (total = 6)

Two-pointer invariant:

  if h[L] <= h[R]:
      right side is AT LEAST as tall as left_max
      -> water ceiling is left_max (the shorter side)
      -> water at L = left_max - h[L]
      -> move L right
  else:
      left side is AT LEAST as tall as right_max
      -> water ceiling is right_max
      -> water at R = right_max - h[R]
      -> move R left

The key: you never need to scan the whole array
to find the max on one side — the OTHER pointer
already guarantees the taller wall exists.
```

## When To Use This Pattern

- When you see **water trapped between bars** or a
  **min-of-two-maxes** formula, think **two pointers
  with running left_max and right_max**
- When you know **one side already exceeds the other's
  max**, think **process the shorter side now — the
  taller side guarantees the ceiling**
- When the brute-force needs O(n) arrays to precompute
  max heights, think **can two pointers track that
  in O(1) while squeezing inward?**
- When you see **height[left] <= height[right]**,
  think **left_max is the water ceiling at L**;
  process left and advance L inward

## The Approach

Start with left at index 0, right at the last index,
and both running maxes at zero.
At each step, compare the heights at the two pointers:
whichever side is shorter sets the water ceiling for
that position — the taller side guarantees a wall
tall enough on the other side.
Update the running max for the active side, add the
trapped water (max minus current height), then move
that pointer one step inward.
Stop when the pointers meet.

In [9]:
from typing import List  # type hints for the solution

In [10]:
def test_harness(func):
    """Run test cases for trap."""
    tests = [
        # (height, expected_water)
        ([0,1,0,2,1,0,1,3,2,1,2,1], 6),
        ([4,2,0,3,2,5],              9),
        ([1,0,1],                    1),
        ([3,0,2,0,4],                7),
        ([0,0,0],                    0),
        ([1],                        0),
    ]
    passed = 0
    for i, (height, expected) in enumerate(tests):
        result = func(height)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"Test {i+1}: {status}")
        if status == "FAILED":
            print(f"  Input:    {height}")
            print(f"  Expected: {expected}")
            print(f"  Got:      {result}")
    print(f"\n{passed}/{len(tests)} tests passed.")

In [11]:
def trap(height: List[int]) -> int:
    """
    Compute total water trapped between elevation bars.

    Two pointers from both ends with running left_max
    and right_max. When h[left] <= h[right], right side
    guarantees a wall >= left_max, so water at left =
    left_max - h[left]; advance left. Otherwise mirror
    logic for right. Stop when pointers meet.

    Time:  O(n) — each pointer moves at most n steps
    Space: O(1) — two pointers, two running maxes
    """
    if not height: return 0
    l, r = 0, len(height) -1
    res = 0
    leftMax, rightMax = height[l], height[r]
    while l<r :
        if leftMax <= rightMax:
            l += 1
            leftMax = max(leftMax, height[l])
            res += leftMax - height[l]
        else:
            r -= 1
            rightMax = max(rightMax, height[r])
            res += rightMax - height[r]
    return res
            
    


# Quick debug — run this cell while building
print(trap([0,1,0,2,1,0,1,3,2,1,2,1]))  # 6
print(trap([4,2,0,3,2,5]))             # 9
print(trap([3,0,3]))                     # 3
print(trap([1,2,3,4,5]))                 # 0
test_harness(trap)

6
9
3
0
Test 1: PASSED
Test 2: PASSED
Test 3: PASSED
Test 4: PASSED
Test 5: PASSED
Test 6: PASSED

6/6 tests passed.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(trap)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — scan left and right per bar | O(n²) | O(1) |
| Precompute left_max[] and right_max[] arrays | O(n) | O(n) |
| Two pointers with running max | O(n) | O(1) |

Two pointers matches the precomputed-array approach
in time but drops the space to O(1) — the running
max on each side is always sufficient because the
opposite pointer guarantees the taller wall exists.

## Real World Connection

At Citi, intraday liquidity management works exactly
like rain trapping: cash levels across accounts dip
and peak throughout the trading day, and the treasury
desk must calculate how much buffer capacity is
"trapped" between two large inflows — available to
cover outflows without triggering an overdraft.
The elevation bars map to account balances at each
time step; trapped water maps to available intraday
credit headroom between two peak balance moments.
The two-pointer scan runs on the sorted-time ledger
in O(n) — critical when processing millions of
settlement events before the end-of-day cut-off.
On AWS, the same pattern monitors Lambda concurrency
headroom: the "water" is spare capacity trapped
between two burst-limit peaks, usable before the
next auto-scaling event fires.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra